# Bank Account Fraud (BAF, NeurIPS 2022) — Account-Opening Fraud

| | |
|---|---|
| **Source** | Kaggle `sgpjesus/bank-account-fraud-dataset-neurips-2022` |
| **Origin** | Feedzai + academic (Jesus et al., NeurIPS 2022 Datasets track) — privacy-preserving synthesis of real bank data |
| **License** | CC BY-NC-SA 4.0 |
| **Samples** | 1,000,000 applications per variant (Base used here), 8 months |
| **Features** | 30 income/velocity/device/session features |
| **Labels** | `fraud_bool` (~1.1% fraud) |
| **Caveats** | Monthly granularity only → synthetic within-month times (flagged). NC license: research only. |
| **Production research suitability** | HIGH — modern, realistic, built for fraud-ML benchmarking with fairness metadata. |

In [1]:
import sys, glob
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from prep_utils import RAW, to_unified, dataset_report, numeric_summary, save_clean, save_unified_part

D = RAW / "financial" / "bank_account_fraud"

In [2]:
base = glob.glob(str(D / "**" / "Base.csv"), recursive=True)[0]
df = pd.read_csv(base)
print(df.shape)
df.head(3)

(1000000, 32)


,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,...,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,...,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,...,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0
2,0,0.8,0.996707,9,14,40,0.012316,-1.490386,AB,1095,...,0,200.0,0,INTERNET,22.730559,windows,0,1,0,0


## Cleaning
BAF encodes missing values as negative sentinels in specific columns (per paper):
`prev_address_months_count`, `current_address_months_count`, `intended_balcon_amount`,
`bank_months_count`, `session_length_in_minutes`, `device_distinct_emails_8w` use -1 (or <0).

In [3]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"dropped {before - len(df)} duplicates")
sentinel_cols = ["prev_address_months_count", "current_address_months_count",
                 "intended_balcon_amount", "bank_months_count",
                 "session_length_in_minutes", "device_distinct_emails_8w"]
for c in sentinel_cols:
    n = int((df[c] < 0).sum())
    if n:
        df[f"{c}_missing"] = (df[c] < 0).astype("int8")
        df.loc[df[c] < 0, c] = np.nan
        print(f"{c}: {n} sentinel-missing -> NaN + indicator")

dropped 0 duplicates
prev_address_months_count: 712920 sentinel-missing -> NaN + indicator
current_address_months_count: 4254 sentinel-missing -> NaN + indicator
intended_balcon_amount: 742523 sentinel-missing -> NaN + indicator
bank_months_count: 253635 sentinel-missing -> NaN + indicator
session_length_in_minutes: 2015 sentinel-missing -> NaN + indicator
device_distinct_emails_8w: 359 sentinel-missing -> NaN + indicator


In [4]:
for c in ["payment_type", "employment_status", "housing_status", "source", "device_os"]:
    df[c] = df[c].astype("category")
assert df["fraud_bool"].isin([0, 1]).all()
df["fraud_bool"].value_counts(normalize=True)

fraud_bool
0    0.988971
1    0.011029
Name: proportion, dtype: float64

## Timestamp normalization
Only `month` (0-7). Anchor 8-month window at 2022-01-01, uniform within month, flagged.

In [5]:
rng = np.random.default_rng(42)
anchor = pd.Timestamp("2022-01-01", tz="UTC")
df["event_time"] = (anchor + pd.to_timedelta(df["month"] * 30, unit="D")
                    + pd.to_timedelta(rng.uniform(0, 30 * 86400, len(df)), unit="s"))
df = df.sort_values("event_time").reset_index(drop=True)

## EDA

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df.groupby("month")["fraud_bool"].mean().plot.bar(ax=axes[0], title="fraud rate by month")
df.groupby("device_os", observed=True)["fraud_bool"].mean().plot.bar(ax=axes[1], title="fraud rate by device OS")
axes[2].hist(df["credit_risk_score"].dropna(), bins=60); axes[2].set_title("credit_risk_score")
plt.tight_layout(); plt.savefig("../reports/baf_eda.png", dpi=110); plt.show()

/tmp/ipykernel_188774/1909575887.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("../reports/baf_eda.png", dpi=110); plt.show()


In [7]:
numeric_summary(df, "bank_account_fraud").head(12)

,count,mean,std,min,25%,50%,75%,max
fraud_bool,1000000.0,0.011029,0.104438,0.000000e+00,0.000000,0.000000,0.000000,1.000000
income,1000000.0,0.562696,0.290343,1.000000e-01,0.300000,0.600000,0.800000,0.900000
name_email_similarity,1000000.0,0.493694,0.289125,1.434550e-06,0.225216,0.492153,0.755567,0.999999
prev_address_months_count,287080.0,60.719967,63.578187,5.000000e+00,25.000000,34.000000,72.000000,383.000000
current_address_months_count,995746.0,86.962058,88.409289,0.000000e+00,20.000000,53.000000,130.000000,428.000000
customer_age,1000000.0,33.689080,12.025799,1.000000e+01,20.000000,30.000000,40.000000,90.000000
days_since_request,1000000.0,1.025705,5.381835,4.036860e-09,0.007193,0.015176,0.026331,78.456904
intended_balcon_amount,257477.0,36.582496,23.236885,5.428451e-05,20.403236,32.433701,49.586253,112.956928
zip_count_4w,1000000.0,1572.692049,1005.374565,1.000000e+00,894.000000,1263.000000,1944.000000,6700.000000
velocity_6h,1000000.0,5665.296605,3009.380665,-1.706031e+02,3436.365848,5319.769349,7680.717827,16715.565404


## Save clean + unified

In [8]:
save_clean(df, "bank_account_fraud")
dataset_report(df, "bank_account_fraud", label_col="fraud_bool",
               notes="Sentinel -1 -> NaN + indicators; month -> synthetic timestamps anchored 2022-01.")

clean -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/data/clean/bank_account_fraud.parquet (1,000,000 rows)


report -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/reports/bank_account_fraud_stats.json


{'dataset': 'bank_account_fraud',
 'rows': 1000000,
 'columns': 39,
 'memory_mb': 235.0,
 'duplicate_rows': 0,
 'missing_by_column': {'prev_address_months_count': 712920,
  'current_address_months_count': 4254,
  'intended_balcon_amount': 742523,
  'bank_months_count': 253635,
  'session_length_in_minutes': 2015,
  'device_distinct_emails_8w': 359},
 'dtypes': {'fraud_bool': 'int64',
  'income': 'float64',
  'name_email_similarity': 'float64',
  'prev_address_months_count': 'float64',
  'current_address_months_count': 'float64',
  'customer_age': 'int64',
  'days_since_request': 'float64',
  'intended_balcon_amount': 'float64',
  'payment_type': 'category',
  'zip_count_4w': 'int64',
  'velocity_6h': 'float64',
  'velocity_24h': 'float64',
  'velocity_4w': 'float64',
  'bank_branch_count_8w': 'int64',
  'date_of_birth_distinct_emails_4w': 'int64',
  'employment_status': 'category',
  'credit_risk_score': 'int64',
  'email_is_free': 'int64',
  'housing_status': 'category',
  'phone_home

In [9]:
u = pd.DataFrame({
    "event_time": df["event_time"],
    "event_subtype": "application",
    "amount": df["proposed_credit_limit"],
    "duration_s": df["session_length_in_minutes"] * 60,
    "severity": np.where(df["fraud_bool"] == 1, 3, 0).astype("int8"),
    "label": df["fraud_bool"].astype("Int8"),
    "time_is_synthetic": True,
})
attr_cols = ["income", "customer_age", "employment_status", "payment_type", "housing_status",
             "credit_risk_score", "velocity_6h", "velocity_24h", "velocity_4w",
             "device_os", "email_is_free", "foreign_request", "keep_alive_session",
             "phone_home_valid", "phone_mobile_valid", "month"]
u[attr_cols] = df[attr_cols]
u = to_unified(u, source_dataset="baf", event_domain="financial",
               event_type="account_open", label_type="fraud", attributes_cols=attr_cols)
save_unified_part(u, "baf")
u.head(3)

unified part -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/data/unified/part_baf.parquet (1,000,000 rows)


,event_id,event_time,event_domain,event_type,event_subtype,source_dataset,user_id,device_id,src_ip,dst_ip,...,amount,duration_s,bytes_in,bytes_out,severity,label,label_type,attack_technique,time_is_synthetic,attributes
0,baf-0,2022-01-01 00:00:34.132897690+00:00,financial,account_open,application,baf,<NA>,<NA>,<NA>,<NA>,...,1500.0,258.282970,NaN,NaN,0,0,fraud,<NA>,True,"{""income"":0.9,""customer_age"":30,""employment_st..."
1,baf-1,2022-01-01 00:00:34.427735075+00:00,financial,account_open,application,baf,<NA>,<NA>,<NA>,<NA>,...,200.0,940.079146,NaN,NaN,0,0,fraud,<NA>,True,"{""income"":0.1,""customer_age"":30,""employment_st..."
2,baf-2,2022-01-01 00:00:49.868826972+00:00,financial,account_open,application,baf,<NA>,<NA>,<NA>,<NA>,...,200.0,116.131221,NaN,NaN,0,0,fraud,<NA>,True,"{""income"":0.7000000000000001,""customer_age"":30..."
